In [2]:
import numpy as np
import pandas as pd

In [3]:
data = pd.read_csv("Hospital_RawDataset_Updated.csv")

In [5]:
print("Number of rows:", len(data))
print("Number of columns:", len(data.columns))

# Show the first 5 rows
data.head()

Number of rows: 10000
Number of columns: 49


,Hospital ID,Hospital Name,Hospital Type,City,State,Department,Department ID,Doctor,Nurses,Staff Count,...,Transferred,Transfer_From_Department,Transfer_To_Department,Transfer_Date,Number_of_Transfers,Dept_ICU_Bed_Capacity_Derived,Dept_Staff_Capacity_Derived,Admissions_Rate_%_Derived,Staff_Utilization_%_Derived,Bed_Occupancy_Rate_%
0,HOSP00016,C K Hospital,Private,Mumbai,Maharashtra,General Medicine,DEPT004,Dr. Arjun Sharma,12,257,...,No,General Medicine,NaN,NaN,0,98,598,0.0,43.0,0.2
1,HOSP00019,Cure Well Hospital,Private,Bengaluru,Karnataka,General Surgery,DEPT010,Dr. Sneha Rao,15,288,...,No,General Surgery,NaN,NaN,0,100,583,0.2,49.4,0.2
2,HOSP00004,Yashoda Hospital,Private,Hyderabad,Telangana,Pulmonology,DEPT005,Dr. Priya Reddy,20,181,...,No,Pulmonology,NaN,NaN,0,97,590,0.0,30.7,0.2
3,HOSP00017,Dharma Hospital,Private,Delhi,Delhi,Pulmonology,DEPT005,Dr. Rahul Verma,23,170,...,Yes,Pulmonology,Cardiology,9/17/2025,1,99,597,0.2,28.5,0.4
4,HOSP00017,Dharma Hospital,Private,Delhi,Delhi,ICU,DEPT009,Dr. Priya Reddy,34,116,...,No,ICU,NaN,NaN,0,100,576,0.0,20.1,0.6


In [6]:
# Count how many duplicate rows exist
num_duplicates = data.duplicated().sum()
print("Number of duplicate rows:", num_duplicates)

# Count how many Patient IDs are repeated
num_duplicate_ids = data["Patient ID"].duplicated().sum()
print("Number of duplicate Patient IDs:", num_duplicate_ids)

Number of duplicate rows: 0
Number of duplicate Patient IDs: 0


In [7]:
# Remove duplicate rows (if any)
data = data.drop_duplicates()

# Remove rows with a repeated Patient ID, keep the first one
data = data.drop_duplicates(subset="Patient ID", keep="first")

print("Rows remaining after removing duplicates:", len(data))

Rows remaining after removing duplicates: 10000


In [8]:
data["Admission Date"] = pd.to_datetime(data["Admission Date"])
data["Discharge Date"] = pd.to_datetime(data["Discharge Date"])

print("Admission Date column type:", data["Admission Date"].dtype)
print("Discharge Date column type:", data["Discharge Date"].dtype)

Admission Date column type: datetime64[ns]
Discharge Date column type: datetime64[ns]


In [9]:
# Find rows where this problem exists
problem_rows = data["Discharge Date"] < data["Admission Date"]

print("Number of rows with this problem:", problem_rows.sum())

Number of rows with this problem: 2964


In [10]:
# For those rows, swap Admission Date and Discharge Date

# Step 1: temporarily save the Admission Date values for the problem rows
temp_admission_dates = data.loc[problem_rows, "Admission Date"]

# Step 2: put Discharge Date into Admission Date
data.loc[problem_rows, "Admission Date"] = data.loc[problem_rows, "Discharge Date"]

# Step 3: put the saved value into Discharge Date
data.loc[problem_rows, "Discharge Date"] = temp_admission_dates

# Check that the problem is fixed
problem_rows_after = data["Discharge Date"] < data["Admission Date"]
print("Number of rows with this problem now:", problem_rows_after.sum())

Number of rows with this problem now: 0


In [11]:
data["Length of Stay"] = (data["Discharge Date"] - data["Admission Date"]).dt.days

print("Smallest Length of Stay:", data["Length of Stay"].min())
print("Largest Length of Stay:", data["Length of Stay"].max())
print("Average Length of Stay:", round(data["Length of Stay"].mean(), 1))

Smallest Length of Stay: 1
Largest Length of Stay: 338
Average Length of Stay: 78.8


In [12]:
capacity = data["Dept_Bed_Capacity_Derived"]
available = data["Beds Available"]

occupied_percentage = (capacity - available) / capacity * 100

# Make sure no value is below 0% or above 100%
occupied_percentage = occupied_percentage.clip(lower=0, upper=100)

data["Bed_Occupancy_Rate_%"] = round(occupied_percentage, 2)

print("Smallest occupancy rate:", data["Bed_Occupancy_Rate_%"].min())
print("Largest occupancy rate:", data["Bed_Occupancy_Rate_%"].max())

Smallest occupancy rate: 0.0
Largest occupancy rate: 96.0


In [13]:
print("Missing values before fixing:")
print(data.isnull().sum()[data.isnull().sum() > 0])

Missing values before fixing:
Insurance Provider        2463
Transfer_To_Department    8240
Transfer_Date             8240
dtype: int64


In [14]:
# Fill missing Insurance Provider with a clear label
data["Insurance Provider"] = data["Insurance Provider"].fillna("Self-Pay / Unknown")

# Fill missing transfer department columns
# (these are blank when a patient was never transferred, which is normal)
data["Transfer_To_Department"] = data["Transfer_To_Department"].fillna("Not Transferred")
data["Transfer_From_Department"] = data["Transfer_From_Department"].fillna("Not Transferred")

print("Missing values after fixing:")
print(data.isnull().sum()[data.isnull().sum() > 0])

Missing values after fixing:
Transfer_Date    8240
dtype: int64


In [15]:
print("Final number of rows:", len(data))
print("Final number of columns:", len(data.columns))
print()
print("Negative Length of Stay values:", (data["Length of Stay"] < 0).sum())
print("Occupancy rates above 100 or below 0:",
      ((data["Bed_Occupancy_Rate_%"] < 0) | (data["Bed_Occupancy_Rate_%"] > 100)).sum())
print("Duplicate Patient IDs:", data["Patient ID"].duplicated().sum())

Final number of rows: 10000
Final number of columns: 49

Negative Length of Stay values: 0
Occupancy rates above 100 or below 0: 0
Duplicate Patient IDs: 0


In [16]:
data.to_csv("hospital_cleaned.csv", index=False)
print("Saved as hospital_cleaned.csv")

Saved as hospital_cleaned.csv
